
# Building Energy Efficiency Prediction

## Notebook 02: Data Preparation and Model Training

### Objective

In this notebook, I will prepare the Energy Efficiency
dataset for machine learning and develop regression
models to predict building heating and cooling loads.

### Tasks

1. Load the dataset prepared in Notebook 01.
2. Separate input features and target variables.
3. Identify appropriate preprocessing techniques.
4. Split the dataset into training and testing sets.
5. Establish a baseline prediction model.
6. Train multiple regression models.
7. Evaluate models using MAE, RMSE, and R².
8. Compare model performance for heating and cooling loads.
9. Identify promising models for optimisation in Notebook 03.

### Expected Outcome

I aim to identify regression models that can predict
building heating and cooling requirements from
architectural characteristics.

The best-performing models will undergo further
optimisation and evaluation in Notebook 03.


In [4]:

# Import required libraries

# Data handling

import pandas as pd

# Data preparation and validation
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Regression models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor



In [5]:
# Load the dataset from Notebook 01

df = pd.read_csv("../data/energy_efficiency_renamed.csv")

# Confirm dataset loading

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

# Display the first five rows

display(df.head())


Dataset loaded successfully!
Dataset Shape: (768, 10)


,Relative_Compactness,Surface_Area,Wall_Area,Roof_Area,Overall_Height,Orientation,Glazing_Area,Glazing_Area_Distribution,Heating_Load,Cooling_Load
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


In [6]:
X = df.drop(columns=["Heating_Load", "Cooling_Load"])
y = df[["Heating_Load", "Cooling_Load"]]

print("Features:", X.shape)
print("Targets:", y.shape)

Features: (768, 8)
Targets: (768, 2)


In [7]:

# Check input feature quality
print("Missing values:", X.isnull().sum().sum())
print("Duplicate feature rows:", X.duplicated().sum())

# Inspect categorical features
print("Orientation:", sorted(X["Orientation"].unique()))
print("Glazing Distribution:", sorted(X["Glazing_Area_Distribution"].unique()))


Missing values: 0
Duplicate feature rows: 0
Orientation: [np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Glazing Distribution: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [8]:

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (614, 8)
X_test: (154, 8)
y_train: (614, 2)
y_test: (154, 2)



## Feature Preprocessing

I prepared numerical and categorical features for machine learning using scaling and one-hot encoding.


In [9]:

# Identify feature types
categorical_features = ["Orientation", "Glazing_Area_Distribution"]

numerical_features = [
    col for col in X.columns if col not in categorical_features
]

# Create preprocessing steps
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)
print("Preprocessor created successfully!")


Numerical Features: ['Relative_Compactness', 'Surface_Area', 'Wall_Area', 'Roof_Area', 'Overall_Height', 'Glazing_Area']
Categorical Features: ['Orientation', 'Glazing_Area_Distribution']
Preprocessor created successfully!



## Baseline Model

I will now establish a baseline using DummyRegressor to measure how much my machine learning models improve prediction accuracy.


In [11]:
# Create the baseline model

baseline_model = DummyRegressor(strategy="mean")

# Evaluate each target using cross validation
for target in y_train.columns:
    scores = cross_validate(
        baseline_model, X_train, y_train[target], cv=5, scoring={"MAE": "neg_mean_absolute_error", "R2": "r2"}
    )
    mean_mae = -scores["test_MAE"].mean()
    mean_r2 = scores["test_R2"].mean()
    print(f"Baseline Model MAE for {target}: {mean_mae:.4f}")
    print(f"Baseline Model R2 for {target}: {mean_r2:.4f}")

Baseline Model MAE for Heating_Load: 9.1092
Baseline Model R2 for Heating_Load: -0.0043
Baseline Model MAE for Cooling_Load: 8.5366
Baseline Model R2 for Cooling_Load: -0.0018



## Linear Regression

I will train a Linear Regression model to predict heating and cooling loads and compare its performance against the baseline.


In [12]:
# Create LInear Regression model pipeline

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Evaluate Linear Regression model 
linear_results = {}

for target in y_train.columns:
    scores = cross_validate(
        linear_model, X_train, y_train[target], cv=5, scoring={"MAE": "neg_mean_absolute_error", "R2": "r2"}
    )
    mean_mae = -scores["test_MAE"].mean()
    mean_r2 = scores["test_R2"].mean()
    linear_results[target] = {"MAE": mean_mae, "R2": mean_r2}
    print(f"Linear Regression Model MAE for {target}: {mean_mae:.4f}")
    print(f"Linear Regression Model R2 for {target}: {mean_r2:.4f}")

Linear Regression Model MAE for Heating_Load: 2.0291
Linear Regression Model R2 for Heating_Load: 0.9205
Linear Regression Model MAE for Cooling_Load: 2.3086
Linear Regression Model R2 for Cooling_Load: 0.8832



## Decision Tree Regression

I will train a Decision Tree model to capture nonlinear relationships between building characteristics and energy loads.

I will compare its performance with Linear Regression using cross-validation.


In [13]:
# Create  Decision Tree model pipeline
tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

# Evaluate Decision Tree model
tree_results = {}

for target in y_train.columns:
    scores = cross_validate(
        tree_model, X_train, y_train[target], cv=5, scoring={"MAE": "neg_mean_absolute_error", "R2": "r2"}
    )
    mean_mae = -scores["test_MAE"].mean()
    mean_r2 = scores["test_R2"].mean()
    tree_results[target] = {"MAE": mean_mae, "R2": mean_r2}
    print(f"Decision Tree Model MAE for {target}: {mean_mae:.4f}")
    print(f"Decision Tree Model R2 for {target}: {mean_r2:.4f}")


Decision Tree Model MAE for Heating_Load: 0.3924
Decision Tree Model R2 for Heating_Load: 0.9963
Decision Tree Model MAE for Cooling_Load: 1.2656
Decision Tree Model R2 for Cooling_Load: 0.9392



## Random Forest Regression

I will train a Random Forest model to improve prediction performance and compare its results with Linear Regression and Decision Tree.


In [14]:
# Create Ramdon Forest model pipeline
forest_model = Pipeline([
    ("preprocessor", preprocessor), 
    ("model", RandomForestRegressor(random_state=42))
])

# Evaluate Random Forest model
forest_results = {}

for target in y_train.columns:
    scores = cross_validate(
        forest_model, X_train, y_train[target], cv=5, scoring={"MAE": "neg_mean_absolute_error", "R2": "r2"}
    )
    mean_mae = -scores["test_MAE"].mean()
    mean_r2 = scores["test_R2"].mean()
    forest_results[target] = {"MAE": mean_mae, "R2": mean_r2}
    print(f"Random Forest Model MAE for {target}: {mean_mae:.4f}")
    print(f"Random Forest Model R2 for {target}: {mean_r2:.4f}")

Random Forest Model MAE for Heating_Load: 0.3571
Random Forest Model R2 for Heating_Load: 0.9971
Random Forest Model MAE for Cooling_Load: 1.0721
Random Forest Model R2 for Cooling_Load: 0.9648



## Gradient Boosting Regression

I will train a Gradient Boosting model to investigate whether sequential tree learning improves heating and cooling load predictions compared with previous models.


In [15]:

# Create Gradient Boosting pipeline
boost_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

# Evaluate Gradient Boosting
boost_results = {}

for target in y_train.columns:
    scores = cross_validate(
        boost_model,
        X_train,
        y_train[target],
        cv=5,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "R2": "r2"
        }
    )

    mean_mae = -scores["test_MAE"].mean()
    mean_r2 = scores["test_R2"].mean()

    boost_results[target] = {"MAE": mean_mae, "R2": mean_r2}

    print(f"Gradient Boosting MAE for {target}: {mean_mae:.4f}")
    print(f"Gradient Boosting R2 for {target}: {mean_r2:.4f}\n")


Gradient Boosting MAE for Heating_Load: 0.3401
Gradient Boosting R2 for Heating_Load: 0.9976

Gradient Boosting MAE for Cooling_Load: 1.0451
Gradient Boosting R2 for Cooling_Load: 0.9726




## Model Performance Comparison

I will compare the regression models using cross-validation MAE and R² scores to identify the most promising models for further optimisation.


In [16]:
# Recalculate and store baseline results
baseline_results = {}

for target in y_train.columns:
    scores = cross_validate(
        baseline_model,
        X_train,
        y_train[target],
        cv=5,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "R2": "r2"
        }
    )
    baseline_results[target] = {
        "MAE": -scores["test_MAE"].mean(),
        "R2": scores["test_R2"].mean()
    }

# combine model results
model_results = {
    "Baseline": baseline_results,
    "Linear Regression": linear_results,
    "Decision Tree": tree_results,
    "Random Forest": forest_results,
    "Gradient Boosting": boost_results
}

# Create comparison table 
comparison_table = pd.DataFrame([
    {"Model":model, "Target":target, **metrics}
    for model, results in model_results.items()
    for target, metrics in results.items()
])

display(comparison_table.round(4))

,Model,Target,MAE,R2
0,Baseline,Heating_Load,9.1092,-0.0043
1,Baseline,Cooling_Load,8.5366,-0.0018
2,Linear Regression,Heating_Load,2.0291,0.9205
3,Linear Regression,Cooling_Load,2.3086,0.8832
4,Decision Tree,Heating_Load,0.3924,0.9963
5,Decision Tree,Cooling_Load,1.2656,0.9392
6,Random Forest,Heating_Load,0.3571,0.9971
7,Random Forest,Cooling_Load,1.0721,0.9648
8,Gradient Boosting,Heating_Load,0.3401,0.9976
9,Gradient Boosting,Cooling_Load,1.0451,0.9726



### Model Comparison Findings

- I evaluated five regression models using 5-fold cross-validation.
- Gradient Boosting achieved the lowest MAE and highest R² for both targets.
- Heating Load: MAE = 0.3401, R² = 0.9976.
- Cooling Load: MAE = 1.0451, R² = 0.9726.
- I selected Gradient Boosting for further optimisation in Notebook 03.


In [18]:

# Save model comparison results
comparison_table.to_csv("../data/model_comparison.csv", index=False)

print("Model comparison saved successfully!")


Model comparison saved successfully!
